In [ ]:
immport pandas as pd
impot numpy as np
import matplotlib as plt

Load in data set using AWS- this is a bigger data source than the synthetic medical data we used in a previous homework.

In [ ]:
!apt-get install awscli -q
!aws s3 cp s3://synthea-omop/synthea100k/ ./synthea100k/ --recursive --no-sign-request

Important Step to Avoid Parsing Issues.

In [ ]:
!apt-get install -y lzop
!lzop -d synthea100k/*.lzo

In [ ]:
visits = pd.read_csv("synthea100k/visit_occurrence.csv")
print(visits.head())
print("Total visits:", len(visits))

In [ ]:
hospital_visits = visits[visits['visit_concept_id'] == 9201].copy()
hospital_visits['visit_start_date'] = pd.to_datetime(hospital_visits['visit_start_date'])
hospital_visits = hospital_visits.sort_values(['person_id', 'visit_start_date'])

In [ ]:
# Get the next visit start per patient
hospital_visits['next_visit_start'] = hospital_visits.groupby('person_id')['visit_start_date'].shift(-1)

# Days until next visit
hospital_visits['DAYS_TO_NEXT'] = (hospital_visits['next_visit_start'] - hospital_visits['visit_start_date']).dt.days

In [ ]:
hospital_visits['READMIT_30'] = hospital_visits['DAYS_TO_NEXT'].apply(
    lambda x: 1 if pd.notnull(x) and x <= 30 else 0
)

In [ ]:
print(hospital_visits['READMIT_30'].value_counts())

In [ ]:
person = pd.read_csv("synthea100k/person.csv")

In [ ]:
print(person['person_id'].nunique())
print(len(person))

In [ ]:
person_clean = person.drop_duplicates(subset=['person_id'])

In [ ]:
data = hospital_visits.merge(person_clean, on='person_id', how='left')

In [ ]:
data['AGE'] = (pd.to_datetime('today') - pd.to_datetime(data['birth_datetime'])).dt.days // 365

In [ ]:
data = data.sort_values(['person_id', 'visit_start_date'])
data['PRIOR_ADMISSIONS'] = data.groupby('person_id').cumcount()

In [ ]:
final_df = data[['person_id', 'READMIT_30', 'AGE', 'gender_concept_id', 'race_concept_id', 'PRIOR_ADMISSIONS']]

In [ ]:
final_df.to_csv("synthetic_readmission_final.csv", index=False)

In [ ]:
from google.colab import files

In [ ]:
files.download("synthetic_readmission_final.csv")